## Notes from Calvin:

### Since we are still working on cleaning our datasets, I am testing this script using dummy data provided by Claude. Annotations are made in each code block to explain (in human language) what the code is doing.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.fake_provider import FakeMarrakesh
from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit.primitives import StatevectorSampler as Sampler
from sklearn.svm import SVC
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import pandas as pd
import numpy as np

backend = FakeMarrakesh()

In [2]:
df = pd.read_csv('../data/processed/wildfire_cleaned_TEST.csv')
df.head()

,zip,Year,avg_tmax_c,avg_tmin_c,tot_prcp_mm,wildfire_occurred,num_fires,total_acres,max_acres,avg_acres,most_common_cause
0,90001,2018,22.319292,14.603904,198.1,0,0.0,0.0,0.0,0.0,0
1,90001,2019,21.559567,13.958907,475.7,0,0.0,0.0,0.0,0.0,0
2,90001,2020,22.298544,14.032893,229.6,0,0.0,0.0,0.0,0.0,0
3,90001,2021,20.878253,13.234925,307.3,0,0.0,0.0,0.0,0.0,0
4,90002,2018,22.319292,14.603904,198.1,0,0.0,0.0,0.0,0.0,0


In [3]:
# In this specific data provided by Claude, it reduced the dataset to 11 columns and recorded a 9:1 ratio imbalance (so a lot more no than yes)

df = df[df["Year"].between(2018, 2022)].reset_index(drop=True)

# Checking for any bad rows that managed to slip past cleaning
assert "wildfire_occurred" in df.columns, "Missing target column!"
assert df["wildfire_occurred"].isin([0, 1]).all(), "Target column has non-binary values!"
assert df.duplicated(subset=["Year", "zip"]).sum() == 0, "Duplicate Year+ZIP rows found!"
assert df["avg_tmax_c"].isnull().sum() == 0, "Null values found in weather features!"

print("Shape:", df.shape)
print("Years:", sorted(df["Year"].unique()))
print("Unique ZIPs:", df["zip"].nunique())
print("Nulls:\n", df.isnull().sum()[df.isnull().sum() > 0])
print("\nTarget distribution:")
print(df["wildfire_occurred"].value_counts())
print(f"Class imbalance ratio: {df['wildfire_occurred'].value_counts()[0] / df['wildfire_occurred'].value_counts()[1]:.1f}:1")

Shape: (10544, 11)
Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Unique ZIPs: 2593
Nulls:
 Series([], dtype: int64)

Target distribution:
wildfire_occurred
0    9498
1    1046
Name: count, dtype: int64
Class imbalance ratio: 9.1:1


In [4]:
df = df.drop(columns=["most_common_cause"]) # Dropping since column is too noisy (too many 0 entries that are insignificant to the model)

df["fire_size_risk"] = df["max_acres"] / (df["avg_acres"] + 1) # Severity relative to avg
df["temp_range"] = df["avg_tmax_c"] - df["avg_tmin_c"] # Daily temperature swing

print("Final features:")
print(df.drop(columns=["wildfire_occurred"]).columns.tolist())
print("\nShape:", df.shape)

Final features:
['zip', 'Year', 'avg_tmax_c', 'avg_tmin_c', 'tot_prcp_mm', 'num_fires', 'total_acres', 'max_acres', 'avg_acres', 'fire_size_risk', 'temp_range']

Shape: (10544, 12)


In [ ]:
# Lagged features must come BEFORE resampling so ZIP-year sequences are intact
df = df.sort_values(["zip", "Year"]).reset_index(drop=True)
df["prev_total_acres"] = df.groupby("zip")["total_acres"].shift(1).fillna(0)
df["prev_num_fires"]   = df.groupby("zip")["num_fires"].shift(1).fillna(0)
df["precip_change"]    = df.groupby("zip")["tot_prcp_mm"].diff().fillna(0)

# Split by year FIRST before any resampling
train = df[df["Year"] <= 2021]
test  = df[df["Year"] == 2022]  # test set never touched by resampling

# Resample only the training data
fire_train    = train[train["wildfire_occurred"] == 1]
no_fire_train = train[train["wildfire_occurred"] == 0]
no_fire_downsampled = resample(no_fire_train, replace=False, n_samples=len(fire_train) * 3, random_state=42)
train = pd.concat([fire_train, no_fire_downsampled]).reset_index(drop=True)

print("Training set after resampling:")
print(train["wildfire_occurred"].value_counts())
print("\nTest set (untouched):")
print(test["wildfire_occurred"].value_counts())

# Exclude current-year fire statistics — they ARE the target and cause data leakage
LEAK_COLS = ["num_fires", "total_acres", "max_acres", "avg_acres", "fire_size_risk"]
FEATURES = [c for c in df.columns if c not in ["zip", "Year", "wildfire_occurred"] + LEAK_COLS]
X_train, y_train = train[FEATURES].values, train["wildfire_occurred"].values
X_test,  y_test  = test[FEATURES].values,  test["wildfire_occurred"].values

print("\nTrain shape:", X_train.shape)
print("Test shape: ", X_test.shape)

# RandomForest to rank feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("\nFeature importances:\n", importances)

TOP_N = 8
top_features = importances.head(TOP_N).index.tolist()
print(f"\nTop {TOP_N} features selected:", top_features)
X_train = train[top_features].values
X_test  = test[top_features].values

# Normalize to [-π, π] for quantum encoding
scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print("\nFeature range after scaling:")
print("  Min:", X_train.min().round(3), " Max:", X_train.max().round(3))

In [ ]:
# This is the actual quantum portion of our code. we're using a ZZFeatureMap, which basically encodes our feature columns into
# their quantum states (converting their values into quantum bits).

NUM_QUBITS = 6

feature_map = ZZFeatureMap(
    feature_dimension = 6, # How many features
    reps = 1, # Repititions
    entanglement = "linear" # How qubits connect/map to each other
)

print(f"Number of qubits : {feature_map.num_qubits}")
print(f"Circuit depth    : {feature_map.decompose().depth()}")
print(f"Number of params : {feature_map.num_parameters}")
print()
feature_map.decompose().draw("text")

# Ok this circuit looks very confusing, but it's just showing the encoding process of our features.

In [ ]:
# A sampler is just the backend that actually runs the quantum circuit. it's just like how we have to select a kernel
# to run Python.
sampler = Sampler()
fidelity = ComputeUncompute(sampler = sampler)

quantum_kernel = FidelityQuantumKernel(
    feature_map = feature_map,
    fidelity = fidelity
)

# SVC stands for Support Vector Classifier. It's a classic ML model which essentially looks at our data and finds the best boundary line
# or (divider) that separates the two classes that we're checking for (in our case, wildfire or no wildfire)
model = SVC(
    kernel = quantum_kernel.evaluate,
    probability = True, # By making this true, we make the model output a probability score instead of just yes/no
    class_weight = 'balanced',
    random_state = 42
)

In [ ]:
# Check if test set has both classes
print("Test set class distribution:")
print(pd.Series(y_test).value_counts())

# Check if train set has both classes  
print("\nTrain set class distribution:")
print(pd.Series(y_train).value_counts())

# Check if 2022 data actually exists
print("\n2022 rows in df:")
print(df[df["Year"] == 2022]["wildfire_occurred"].value_counts())

# Check train/test overlap
print("\nYears in train:", sorted(train["Year"].unique()))
print("Years in test:", sorted(test["Year"].unique()))

# Check shapes
print("\nX_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
subset_idx = np.concatenate([
    np.where(y_train == 0)[0][:100],
    np.where(y_train == 1)[0][:100]
])
X_subset = X_train[subset_idx]
y_subset = y_train[subset_idx]

model.fit(X_subset, y_subset)
print(f"Subset accuracy: {model.score(X_subset, y_subset):.3f}")

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Test Accuracy:", model.score(X_test, y_test))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, labels=[0, 1], target_names=["No Wildfire", "Wildfire"]))
if len(np.unique(y_test)) > 1:
    print("\nAUC-ROC:", roc_auc_score(y_test, y_prob).round(3))
else:
    print("\nAUC-ROC: N/A (test set has only one class)")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))